# WildChat annotations - inspection & recipes

Annotation is run from `src/annotate.py` (`python src/annotate.py --dry-run` /
`--max-batches N`); this notebook only **reads** the annotation store under
`data/wildchat-annotations/shards`. The synthetic pipeline (thin driver in its own notebook,
machinery in `src/synthetic.py` + `src/lib/`) writes the same record schema into
`data/synthetic-annotations/shards`, so both concatenate cleanly when assembling the final
train/test dataset.


In [ ]:
import sys
from pathlib import Path

# make `src.*` imports work no matter where the kernel was started
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.lib.store import AnnotationStore
from src.annotate import PROJECT_ROOT, SHARD_DIR, SETTINGS, FEATURES

store = AnnotationStore(SHARD_DIR, FEATURES)
annotated = store.dataset()
print(f"{len(store)} annotated messages in {store.num_shards} shards "
      f"under {SHARD_DIR.relative_to(PROJECT_ROOT)}")
annotated


In [ ]:
# health check: failed vs clean records
errored = annotated.filter(lambda e: e != "", input_columns="error")
print(f"{errored.num_rows} of {annotated.num_rows} records carry request errors")


In [ ]:
# per-role mean scores across all annotation themes (records with failures removed)
clean = annotated.filter(lambda e: e == "", input_columns="error")
by_role = clean.to_pandas().groupby("role")[SETTINGS.themes].mean()
by_role


### Recipes

```python
# reconstruct one conversation, annotations aligned to message order:
# conv_id = clean[0]["conversation_id"]
# conv = clean.filter(lambda cid: cid == conv_id, input_columns="conversation_id").sort("message_idx")

# a message counts as positive for a theme when its score >= the packaged cutoff:
# from llm_delusions_annotations.cutoffs import load_llm_cutoffs_from_csv
# cutoffs = load_llm_cutoffs_from_csv(None)

# WildChat contains duplicated conversations -> drop duplicate keys if it matters:
# deduped = clean.to_pandas().drop_duplicates(subset=["conversation_id", "message_idx", "content_hash"])

# persist one merged copy for consumers that don't want to touch the shard dir:
# clean.save_to_disk(PROJECT_ROOT / "data" / "wildchat-annotations-compiled")
# (see compile.ipynb for the shard-merging step used later)
```
